In [34]:
# Copyright (c) Meta Platforms, Inc. and affiliates.

# SAM 3 Agent

This notebook shows an example of how an MLLM can use SAM 3 as a tool, i.e., "SAM 3 Agent", to segment more complex text queries such as "the leftmost child wearing blue vest".

## Env Setup

First install `sam3` in your environment using the [installation instructions](https://github.com/facebookresearch/sam3?tab=readme-ov-file#installation) in the repository.

In [35]:
import torch
# turn on tfloat32 for Ampere GPUs
# https://pytorch.org/docs/stable/notes/cuda.html#tensorfloat-32-tf32-on-ampere-devices
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# use bfloat16 for the entire notebook. If your card doesn't support it, try float16 instead
torch.autocast("cuda", dtype=torch.bfloat16).__enter__()

# inference mode for the whole notebook. Disable if you need gradients
torch.inference_mode().__enter__()

In [36]:
import os

SAM3_ROOT = os.path.dirname(os.getcwd())
os.chdir(SAM3_ROOT)

# setup GPU to use -  A single GPU is good with the purpose of this demo
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
_ = os.system("nvidia-smi")

Sun Mar 29 16:19:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.07             Driver Version: 581.80         CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4070 Ti     On  |   00000000:01:00.0  On |                  N/A |
|  0%   39C    P8             13W /  285W |   11815MiB /  12282MiB |     11%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Build SAM3 Model

In [37]:
import sam3
from sam3 import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

sam3_root = os.path.dirname(sam3.__file__)
bpe_path = f"{sam3_root}/assets/bpe_simple_vocab_16e6.txt.gz"
model = build_sam3_image_model(bpe_path=bpe_path)
processor = Sam3Processor(model, confidence_threshold=0.5)

## LLM Setup

Config which MLLM to use, it can either be a model served by vLLM that you launch from your own machine or a model is served via external API. If you want to using a vLLM model, we also provided insturctions below.

In [38]:
LLM_CONFIGS = {
    # vLLM-served models
    "qwen3_vl_8b_thinking": {
        "provider": "vllm",
        "model": "Qwen/Qwen3-VL-8B-Thinking",
    },
    # models served via external APIs
    # add your own
}

model = "qwen3_vl_8b_thinking"
LLM_API_KEY = "DUMMY_API_KEY"

llm_config = LLM_CONFIGS[model]
llm_config["api_key"] = LLM_API_KEY
llm_config["name"] = model

# setup API endpoint
if llm_config["provider"] == "vllm":
    # Use 127.0.0.1 for local client requests; 0.0.0.0 is for server bind only.
    LLM_SERVER_URL = os.environ.get("LLM_SERVER_URL", "http://127.0.0.1:8001/v1")
else:
    LLM_SERVER_URL = llm_config["base_url"]

### Setup vLLM server 
This step is only required if you are using a model served by vLLM, skip this step if you are calling LLM using an API like Gemini and GPT.

* Install vLLM (in a separate conda env from SAM 3 to avoid dependency conflicts).
  ```bash
    conda create -n vllm python=3.12
    pip install vllm --extra-index-url https://download.pytorch.org/whl/cu128
  ```
* Start vLLM server on the same machine as this notebook
  ```bash
    # qwen 3 VL 8B thinking (single-GPU example)
    vllm serve Qwen/Qwen3-VL-8B-Thinking --tensor-parallel-size 1 --allowed-local-media-path / --enforce-eager --port 8001
  ```
* If you have multiple GPUs, set `--tensor-parallel-size` to the number of GPUs you want to use.

## Run SAM3 Agent Inference

In [39]:
from functools import partial
from IPython.display import display, Image
from sam3.agent.client_llm import send_generate_request as send_generate_request_orig
from sam3.agent.client_sam3 import call_sam_service as call_sam_service_orig
from sam3.agent.inference import run_single_image_inference

In [40]:
# prepare input args and run single image inference
import requests

image_candidates = [
    os.path.join(sam3_root, "assets", "images", "test_image.jpg"),
    os.path.join(os.path.dirname(sam3_root), "assets", "images", "test_image.jpg"),
]
image = next((os.path.abspath(p) for p in image_candidates if os.path.exists(p)), image_candidates[0])
prompt = "the leftmost child wearing blue vest"

server_ok = True
if llm_config["provider"] == "vllm":
    try:
        _ = requests.get(f"{LLM_SERVER_URL}/models", timeout=5)
    except Exception:
        server_ok = False
        print(
            f"Cannot reach vLLM server at {LLM_SERVER_URL}. "
            "Start the server first (e.g. `vllm serve ... --port 8001`) "
            "or set LLM_SERVER_URL to a reachable endpoint."
        )

output_image_path = None
if server_ok:
    send_generate_request = partial(
        send_generate_request_orig,
        server_url=LLM_SERVER_URL,
        model=llm_config["model"],
        api_key=llm_config["api_key"],
    )
    call_sam_service = partial(call_sam_service_orig, sam3_processor=processor)
    output_image_path = run_single_image_inference(
        image, prompt, llm_config, send_generate_request, call_sam_service,
        debug=True, output_dir="agent_output"
    )

# display output
if output_image_path is not None:
    display(Image(filename=output_image_path))

Cannot reach vLLM server at http://127.0.0.1:8001/v1. Start the server first (e.g. `vllm serve ... --port 8001`) or set LLM_SERVER_URL to a reachable endpoint.
